# Recursividade: funções que chamam a si mesmas e a pilha de execução — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

Este tutorial retoma o programa `pilha.c` visto em aula: um fatorial que narra a própria ida e
volta, uma função que imprime antes e depois de chamar a si mesma, um Fibonacci que conta as
próprias chamadas e uma soma de vetor. Depois de cinco blocos de prática vêm as tarefas de
modificação e o desafio.

## Objetivos

Ao final deste tutorial você será capaz de:

- Identificar o caso base e o passo recursivo de uma função e garantir que o passo se aproxima do
  caso base;
- Explicar o que é um registro de ativação e por que cada chamada recursiva tem as suas próprias
  variáveis locais;
- Prever a ordem da saída de uma função recursiva a partir da posição do código em relação à
  chamada (antes: na ida; depois: na volta);
- Desenhar a árvore de chamadas de uma recursão múltipla e distinguir *total de chamadas* de
  *chamadas simultâneas na pilha*;
- Reescrever uma recursão simples como laço e dizer o que cada versão custa.

In [ ]:
# Confira se o gcc está disponível no seu ambiente
!gcc --version | head -1

## O programa da aula

Recompile e execute o programa condutor **exatamente como saiu da aula** e confira a saída com a
do slide. Os endereços impressos depois de `&n =` serão outros na sua máquina — compare o
*padrão* entre eles, não os números.

In [ ]:
%%writefile pilha.c
/* pilha.c --- o que acontece a cada chamada de uma funcao recursiva */
#include <stdio.h>

/* A: fatorial, contando o que acontece na ida e na volta */
long fatorial(int n)
{
    static int nivel = 0;   /* static: um so para todas as chamadas */
    printf("%*sfatorial(%d) comeca   &n = %p\n", 2 * nivel, "", n,
           (void *) &n);
    nivel++;
    long resultado;
    if (n <= 1)
        resultado = 1;                      /* caso base */
    else
        resultado = n * fatorial(n - 1);    /* passo recursivo */
    nivel--;
    printf("%*sfatorial(%d) devolve %ld\n", 2 * nivel, "", n, resultado);
    return resultado;
}

/* B: o que se faz antes e depois da chamada */
void desce_e_sobe(int n)
{
    if (n == 0)
        return;
    printf("%d ", n);       /* antes da chamada */
    desce_e_sobe(n - 1);
    printf("%d ", n);       /* depois da chamada */
}

/* C: uma funcao que chama a si mesma duas vezes */
long fib(int n, int *chamadas)
{
    (*chamadas)++;
    if (n < 2)
        return n;
    return fib(n - 1, chamadas) + fib(n - 2, chamadas);
}

/* D: a soma de um vetor e' o ultimo mais a soma do resto */
int soma(const int *v, int n)
{
    if (n == 0)
        return 0;
    return soma(v, n - 1) + v[n - 1];
}

int main(void)
{
    printf("A: 4! = %ld\n", fatorial(4));

    printf("B: ");
    desce_e_sobe(3);
    printf("\n");

    int chamadas = 0;
    long f = fib(5, &chamadas);
    printf("C: fib(5) = %ld com %d chamadas\n", f, chamadas);
    chamadas = 0;
    f = fib(20, &chamadas);
    printf("C: fib(20) = %ld com %d chamadas\n", f, chamadas);

    int v[] = {3, 1, 4, 1, 5};
    printf("D: soma = %d\n", soma(v, 5));
    return 0;
}

In [ ]:
!gcc -Wall -Wextra pilha.c -o pilha && ./pilha
!./pilha | grep -q 'C: fib(20) = 6765 com 21891 chamadas' && echo OK \
  || echo 'Verifique: esperava "C: fib(20) = 6765 com 21891 chamadas"'

## 1. Caso base e passo recursivo

Toda função recursiva tem duas peças:

- o **caso base**: uma instância tão pequena que se resolve *sem* chamar a função;
- o **passo recursivo**: reduz o problema a uma instância **estritamente menor** do mesmo problema,
  chama a função para ela e usa a resposta.

O jeito mais seguro de escrever uma é partir da definição. A soma de um vetor, por exemplo, é
`0` se o vetor é vazio, e *o último elemento mais a soma do resto* se não é. Cada linha da
definição vira uma linha do código — e o `n` diminui de 1 em 1 até chegar ao caso base.

O exemplo abaixo faz o mesmo com a potência $x^n = x \cdot x^{n-1}$, $x^0 = 1$, e mostra o que
acontece com um passo que **não** se aproxima do caso base.

In [ ]:
%%writefile potencia.c
#include <stdio.h>

/* x elevado a n, para n >= 0 */
long potencia(long x, int n)
{
    if (n == 0)                       /* caso base: x^0 = 1 */
        return 1;
    return x * potencia(x, n - 1);    /* passo: x^n = x * x^(n-1) */
}

int main(void)
{
    printf("2^10 = %ld\n", potencia(2, 10));
    printf("3^4  = %ld\n", potencia(3, 4));
    printf("7^0  = %ld\n", potencia(7, 0));
    /* potencia(2, -1) nunca chegaria a n == 0: n desceria para -2, -3, ...
       Por isso o contrato diz "n >= 0". O que acontece com a maquina
       nesse caso e' assunto da proxima aula. */
    return 0;
}

In [ ]:
!gcc -Wall -Wextra potencia.c -o potencia && ./potencia

In [ ]:
%%writefile ex1.c
#include <stdio.h>

/* Exercicio 1: devolva a soma dos digitos de n (n >= 0).
   Exemplo: soma_digitos(1234) = 1 + 2 + 3 + 4 = 10.
   TODO: escreva uma versao RECURSIVA, sem laco.
   Dica: n % 10 e' o ultimo digito; n / 10 e' o numero sem ele.
   Qual e' o menor n que voce sabe responder sem recursao? */
int soma_digitos(int n)
{
    /* TODO: implemente aqui */
    return -1;
}

int main(void)
{
    printf("%d %d %d %d\n", soma_digitos(1234), soma_digitos(7),
           soma_digitos(0), soma_digitos(90817));
    return 0;
}

In [ ]:
!gcc -Wall ex1.c -o ex1 && ./ex1
!./ex1 | grep -q '^10 7 0 25$' && echo OK || echo 'Verifique: esperava "10 7 0 25"'

## 2. A pilha de execução

A cada chamada, a máquina reserva um **registro de ativação** (*stack frame*) só daquela chamada,
com os parâmetros, as variáveis locais e o endereço de retorno. Os registros ficam empilhados: o
da chamada mais recente fica no topo e é o primeiro a sair.

É por isso que, em `pilha.c`, cada chamada de `fatorial` tinha um `&n` diferente, e por isso a
distância entre eles era sempre a mesma: registros da mesma função têm o mesmo tamanho. A
exceção era `static int nivel`: uma local `static` não mora no registro — existe uma só para
todas as chamadas.

O programa abaixo mede o tamanho do registro de duas funções recursivas iguais, exceto por uma
variável local de 200 bytes. Cada chamada recebe o endereço de uma local de quem a chamou e
imprime a distância até a sua própria.

In [ ]:
%%writefile registro.c
#include <stdint.h>
#include <stdio.h>

/* distancia, em bytes, entre a local de quem chamou e a minha */
static long distancia(const void *de_quem_chamou, const void *minha)
{
    return (long) ((uintptr_t) de_quem_chamou - (uintptr_t) minha);
}

void magra(int n, const char *anterior)
{
    char marca = 0;
    if (anterior != NULL)
        printf("magra(%d): registro de %ld bytes\n", n,
               distancia(anterior, &marca));
    if (n > 1)
        magra(n - 1, &marca);
}

void gorda(int n, const char *anterior)
{
    char marca = 0;
    char buffer[200];          /* 200 bytes a mais em cada registro */
    buffer[0] = (char) n;
    if (anterior != NULL)
        printf("gorda(%d): registro de %ld bytes (buffer[0] = %d)\n", n,
               distancia(anterior, &marca), buffer[0]);
    if (n > 1)
        gorda(n - 1, &marca);
}

int main(void)
{
    magra(4, NULL);
    gorda(4, NULL);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra registro.c -o registro && ./registro

**Repare em três coisas na saída:**

1. Dentro de cada função, a distância é **a mesma** de uma chamada para a seguinte: o compilador
   decide o tamanho do registro uma vez, ao compilar a função.
2. A `gorda` tem registros pelo menos 200 bytes maiores: as variáveis locais moram no registro.
   Uma função recursiva com um vetor local grande gasta esse vetor **a cada nível**.
3. Os números exatos dependem do compilador, do processador e das opções (`-O0`, `-O2`): por isso
   eles não aparecem em nenhum livro. O que vale em toda parte é o padrão.

In [ ]:
%%writefile ex2.c
#include <stdio.h>

/* Exercicio 2: devolva o MAIOR elemento de v[0..n-1], n >= 1.
   TODO: escreva uma versao RECURSIVA, sem laco.
   Caso base: um vetor de um elemento so'.
   Passo: o maior de v[0..n-1] e' o maior entre v[n-1] e o maior de v[0..n-2].
   Depois de passar no teste, imprima &n dentro da funcao e confira que
   cada chamada tem o seu proprio n. */
int maximo(const int *v, int n)
{
    /* TODO: implemente aqui */
    return 0;
}

int main(void)
{
    int a[] = {3, 9, 2, 7};
    int b[] = {-5};
    int c[] = {-8, -3, -12};
    printf("%d %d %d\n", maximo(a, 4), maximo(b, 1), maximo(c, 3));
    return 0;
}

In [ ]:
!gcc -Wall ex2.c -o ex2 && ./ex2
!./ex2 | grep -q '^9 -5 -3$' && echo OK || echo 'Verifique: esperava "9 -5 -3"'

## 3. A ida e a volta

Em uma função recursiva, a posição do código em relação à chamada decide **quando** ele roda:

- o que vem **antes** da chamada roda na **ida**, na ordem das chamadas;
- o que vem **depois** roda na **volta**, na ordem inversa — quando a chamada de cima retorna e o
  registro de baixo volta a ser o topo, ainda com os seus valores.

Foi assim que `desce_e_sobe(3)` imprimiu `3 2 1 1 2 3`. O exemplo abaixo usa só a volta para
imprimir uma string de trás para frente, sem índice e sem `strlen`.

In [ ]:
%%writefile contrario.c
#include <stdio.h>

void ao_contrario(const char *s)
{
    if (*s == '\0')          /* string vazia: nada a fazer */
        return;
    ao_contrario(s + 1);     /* primeiro, o resto da string... */
    putchar(*s);             /* ...depois, a minha letra (na volta) */
}

void na_ordem(const char *s)
{
    if (*s == '\0')
        return;
    putchar(*s);             /* a minha letra primeiro (na ida) */
    na_ordem(s + 1);
}

int main(void)
{
    ao_contrario("pilha");
    putchar('\n');
    na_ordem("pilha");
    putchar('\n');
    return 0;
}

In [ ]:
!gcc -Wall -Wextra contrario.c -o contrario && ./contrario

In [ ]:
%%writefile ex3.c
#include <stdio.h>

/* Exercicio 3: imprima n (n >= 1) em binario, sem zeros a esquerda.
   Exemplo: binario(13) imprime 1101.
   n % 2 e' o ULTIMO bit, mas ele precisa sair por ULTIMO.
   TODO: escreva uma versao recursiva, sem vetor auxiliar.
   Pense: o putchar vem antes ou depois da chamada? */
void binario(int n)
{
    /* TODO: implemente aqui */
}

int main(void)
{
    binario(13); putchar(' ');
    binario(1);  putchar(' ');
    binario(6);  putchar(' ');
    binario(255);
    putchar('\n');
    return 0;
}

In [ ]:
!gcc -Wall ex3.c -o ex3 && ./ex3
!./ex3 | grep -q '^1101 1 110 11111111$' && echo OK \
  || echo 'Verifique: esperava "1101 1 110 11111111"'

## 4. Recursão múltipla

Quando uma função chama a si mesma **mais de uma vez**, as chamadas formam uma **árvore**. Em
`fib(4)`, a raiz abre `fib(3)` e `fib(2)`; `fib(3)` abre `fib(2)` e `fib(1)`; e assim por diante:
9 chamadas no total, com `fib(2)` calculada duas vezes do zero.

Duas medidas diferentes convivem aqui:

- o **total de chamadas** cresce como a árvore — para `fib`, $2F(n+1) - 1$, quase o dobro do
  próprio resultado;
- as **chamadas ao mesmo tempo na pilha** nunca passam da altura da árvore, $n$: a pilha guarda só
  o caminho da raiz até a chamada atual.

O programa abaixo mede as duas para vários $n$; a célula seguinte desenha o gráfico.

In [ ]:
%%writefile fib_conta.c
#include <stdio.h>

long fib(int n, int nivel, long *chamadas, int *mais_fundo)
{
    (*chamadas)++;
    if (nivel > *mais_fundo)
        *mais_fundo = nivel;
    if (n < 2)
        return n;
    return fib(n - 1, nivel + 1, chamadas, mais_fundo)
         + fib(n - 2, nivel + 1, chamadas, mais_fundo);
}

int main(void)
{
    for (int n = 1; n <= 30; n++) {
        long chamadas = 0;
        int mais_fundo = 0;
        long f = fib(n, 1, &chamadas, &mais_fundo);
        printf("%d %ld %ld %d\n", n, f, chamadas, mais_fundo);
    }
    return 0;
}

In [ ]:
!gcc -Wall -Wextra fib_conta.c -o fib_conta && ./fib_conta | tail -5

In [ ]:
# Visualização: chamadas no total x registros ao mesmo tempo na pilha
import subprocess
import matplotlib.pyplot as plt

linhas = subprocess.run(['./fib_conta'], capture_output=True, text=True).stdout.split('\n')
dados = [tuple(map(int, l.split())) for l in linhas if l.strip()]
ns          = [d[0] for d in dados]
chamadas    = [d[2] for d in dados]
mais_fundo  = [d[3] for d in dados]

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(ns, chamadas, 'o-', label='chamadas no total')
ax.semilogy(ns, mais_fundo, 's-', label='registros ao mesmo tempo (pilha)')
ax.set_xlabel('n')
ax.set_ylabel('quantidade (escala logarítmica)')
ax.set_title('fib(n) ingênua: tempo e pilha crescem de jeitos diferentes')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.show()

Em escala logarítmica, a curva das chamadas é quase uma **reta**: o total se multiplica por
$\approx 1{,}6$ a cada passo de $n$ (crescimento exponencial). A curva da pilha é o próprio $n$ e
mal sai do chão. O problema da `fib` ingênua é **trabalho repetido**, não memória.

In [ ]:
%%writefile ex4.c
#include <stdio.h>

/* Exercicio 4: quantos caminhos existem do canto superior esquerdo ao
   canto inferior direito de uma grade com l linhas e c colunas de passos,
   andando so' para a direita ou para baixo?
   - se l == 0 ou c == 0, ha' um caminho so' (uma linha reta);
   - senao, o primeiro passo e' para baixo OU para a direita.
   TODO: escreva caminhos recursivamente e conte as chamadas em *chamadas. */
long caminhos(int l, int c, long *chamadas)
{
    /* TODO: implemente aqui */
    return 0;
}

int main(void)
{
    long k1 = 0, k2 = 0;
    long a = caminhos(2, 2, &k1);
    long b = caminhos(3, 3, &k2);
    printf("%ld %ld | chamadas: %ld %ld\n", a, b, k1, k2);
    return 0;
}

In [ ]:
!gcc -Wall ex4.c -o ex4 && ./ex4
!./ex4 | grep -q '^6 20 | chamadas: 11 39$' && echo OK \
  || echo 'Verifique: esperava "6 20 | chamadas: 11 39"'

Desenhe a árvore de `caminhos(2, 2)`: quais chamadas se repetem? Quantos registros, no máximo,
ficam na pilha ao mesmo tempo para `caminhos(l, c)`?

## 5. Recursão ou laço?

Toda função recursiva pode ser reescrita com um laço — às vezes com uma pilha feita à mão. A
versão recursiva costuma ser mais próxima da definição; a iterativa usa **um** registro só, em vez
de um por chamada ativa.

Recursão compensa quando o problema já se divide em partes do mesmo tipo (pastas, árvores,
metades de vetor). O laço compensa quando a recursão é só uma contagem disfarçada (fatorial,
soma) ou quando os subproblemas se repetem (a `fib` ingênua).

In [ ]:
%%writefile fatorial2.c
#include <stdio.h>

long fatorial_rec(int n)
{
    if (n <= 1)
        return 1;
    return n * fatorial_rec(n - 1);     /* n registros na pilha */
}

long fatorial_iter(int n)
{
    long r = 1;                         /* um registro so' */
    for (int i = 2; i <= n; i++)
        r *= i;
    return r;
}

int main(void)
{
    for (int n = 0; n <= 20; n += 5)
        printf("%2d! = %20ld  %20ld\n", n, fatorial_rec(n), fatorial_iter(n));
    return 0;
}

In [ ]:
!gcc -Wall -Wextra fatorial2.c -o fatorial2 && ./fatorial2

In [ ]:
%%writefile ex5.c
#include <stdio.h>

/* Exercicio 5: Fibonacci SEM recursao.
   F(0) = 0, F(1) = 1, F(n) = F(n-1) + F(n-2).
   TODO: use um laco e duas variaveis com os dois ultimos valores.
   A versao recursiva faria 331 160 281 chamadas para n = 40;
   quantas iteracoes a sua faz? */
long fib_iter(int n)
{
    /* TODO: implemente aqui */
    return -1;
}

int main(void)
{
    printf("%ld %ld %ld %ld\n", fib_iter(0), fib_iter(1), fib_iter(20),
           fib_iter(40));
    return 0;
}

In [ ]:
!gcc -Wall ex5.c -o ex5 && ./ex5
!./ex5 | grep -q '^0 1 6765 102334155$' && echo OK \
  || echo 'Verifique: esperava "0 1 6765 102334155"'

## Tarefas

Trabalhe sobre uma cópia do programa da aula. A célula abaixo cria `pilha_tarefa.c` a partir de
`pilha.c`; edite esse arquivo (pelo Jupyter ou pelo seu editor) e recompile. **Antes de cada
execução, escreva a saída que você espera.**

**Tarefa 1.** Troque o caso base de `fatorial` para `n == 0` (devolvendo 1). Quantas linhas
`comeca` aparecem agora para `fatorial(4)`? O resultado muda? Chame também `fatorial(0)` nas duas
versões e compare.

**Tarefa 2.** Em `desce_e_sobe`, apague só o primeiro `printf` e preveja a saída; depois restaure-o
e apague só o segundo. Por fim, faça a função imprimir `3 2 1 0 1 2 3` — mexendo no caso base, e
não no `main`.

**Tarefa 3.** Acrescente a `fib` um parâmetro `nivel` e um ponteiro `int *mais_fundo`, como em
`fib_conta.c`. Quantos registros `fib(20)` chega a ter na pilha ao mesmo tempo, contra 21 891
chamadas no total? Explique os dois números com a árvore de chamadas.

In [ ]:
!cp pilha.c pilha_tarefa.c
# edite pilha_tarefa.c e depois rode esta célula de novo (sem o cp, para não apagar sua edição)
!gcc -Wall -Wextra pilha_tarefa.c -o pilha_tarefa && ./pilha_tarefa

## Desafio: potência em poucas chamadas

Escreva duas versões de `long potencia(long x, int n)`, para `n >= 0`:

- **ingênua:** $x^n = x \cdot x^{n-1}$, com $x^0 = 1$ (a da seção 1);
- **rápida:** $x^n = (x^{n/2})^2$ se $n$ é par, e $x \cdot x^{n-1}$ se $n$ é ímpar —
  calculando $x^{n/2}$ **uma vez só** e guardando em uma variável local.

Requisitos:

- as duas devolvem o mesmo valor para $2^0, 2^1, \ldots, 2^{62}$;
- conte as chamadas de cada uma para `potencia(1, 1000)` e explique a diferença;
- desenhe (no papel) a pilha da versão rápida no momento mais fundo de `potencia(2, 10)`: quais
  valores de `n` estão empilhados?
- **por que** é preciso guardar $x^{n/2}$ em uma variável, em vez de escrever
  `potencia(x, n/2) * potencia(x, n/2)`? Conte as chamadas das duas formas.

In [ ]:
%%writefile desafio.c
#include <stdio.h>

/* TODO: versao ingenua; conte as chamadas em *chamadas */
long potencia_ingenua(long x, int n, long *chamadas)
{
    /* TODO: implemente aqui */
    return 0;
}

/* TODO: versao rapida; conte as chamadas em *chamadas */
long potencia_rapida(long x, int n, long *chamadas)
{
    /* TODO: implemente aqui */
    return 0;
}

int main(void)
{
    int erros = 0;
    for (int n = 0; n <= 62; n++) {
        long k1 = 0, k2 = 0;
        if (potencia_ingenua(2, n, &k1) != potencia_rapida(2, n, &k2))
            erros++;
    }
    long k1 = 0, k2 = 0;
    potencia_ingenua(1, 1000, &k1);
    potencia_rapida(1, 1000, &k2);
    printf("erros: %d | chamadas para n = 1000: ingenua %ld, rapida %ld\n",
           erros, k1, k2);
    return 0;
}

In [ ]:
!gcc -Wall desafio.c -o desafio && ./desafio

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. Para esta aula, os pontos de partida
são a seção 4.10 de Kernighan & Ritchie (recursividade), o capítulo 2 de Feofiloff (algoritmos
recursivos), o capítulo 3 de Tenenbaum et al. e, para a pilha de execução vista pela máquina, a
seção 3.7 de Bryant & O'Hallaron.